# v9_b Step 4: Predict reaction products from steroid substrates

Get predicted reactions and products, then filter by detection in deuterium labeled data.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import mass2chem.formula as asari_formula

from mass2chem import formula
from asari.tools import match_features as mf

In [ ]:
# Define the substrates and reagents for the chemical reactions
substrates = {
    'Estradiol': {
        'mz': 506.23596,        # DnCl
            'formula': 'C18H24O2',
            'formula_dict': {'C': 18, 'H': 24, 'O': 2},
            'rtime': 1094, 
            'id': 'Estradiol'
    },
    'Estrone': {
        'mz': 504.22031,
        'formula': 'C18H22O2',
        'formula_dict': {'C': 18, 'H': 22, 'O': 2},
        'rtime': 1099,
        'id': 'Estrone'
    },
    'DHEA': {
        'mz': 536.29414,        # DnHz
        'formula': 'C19H28O2',
        'formula_dict': {'C': 19, 'H': 28, 'O': 2},
        'rtime': 549,
        'id': 'DHEA'
    },
    'Pregnenolone': {
        'mz': 564.32544,
        'formula': 'C21H32O2',
        'formula_dict': {'C': 21, 'H': 32, 'O': 2},
        'rtime': 640,
        'id': 'Pregnenolone'
    },
    'Progesterone': {
        'mz': 562.30979,
        'formula': 'C21H30O2',
        'formula_dict': {'C': 21, 'H': 30, 'O': 2},
        'rtime': 695,
        'id': 'Progesterone'
    }
}

reagents_mz = {
    'DnCl': 233.05105,
    'DnHz': 247.07793,
    'DmPA': 161.08406, 
}

def parse_chemformula_dict_signed(s):
    """
    Parse a chemical formula string with signed elements into a dictionary.
    A sign applies to all following elements until the next sign;
    elements before any sign are treated as positive.
    
    Args:
        s (str): A chemical formula string with signed elements, e.g., '-H5O' or '+H2-C2'.
        
    Returns:
        dict: A dictionary representation of the chemical formula, 
        e.g., '-H5O' -> {'H': -5, 'O': -1}; '+H2-C2' -> {'H': 2, 'C': -2}.
    """
    import re
    formula_dict = {}
    sign = 1
    for token, element, count in re.findall(r'([+-])|([A-Z][a-z]*)(\d*)', s):
        if token:
            sign = -1 if token == '-' else 1
        else:
            formula_dict[element] = formula_dict.get(element, 0) + sign * (int(count) if count else 1)
    return formula_dict

def do_reaction(input_cpd, reaction, idx, prefix):
    '''
    Performs a reaction on an input compound and returns the resulting product's m/z and formula.
    Can do formular dict calculations by formula.parse_chemformula_dict_comprehensive(formula) )
    '''
    product_mz = input_cpd['mz'] + reaction['massDelta_signed']
    product_formula_dict = formula.add_formula_dict(
        input_cpd['formula_dict'], 
        reaction['formula_dict']
        )
    # only valid formula will continue, otherwise return None
    if product_formula_dict is None:
        return None
    else:
        return {
            'id': f'{prefix}_product_{idx}',
            'mz': product_mz,
            'rtime': 0,  # unknown retention time
            'formula_dict': product_formula_dict,
            'formula': formula.dict_to_hill_formula(product_formula_dict),
            'input_cpd': input_cpd,
            'reaction': reaction
        }
        
def multi_step_reactions(input_cpd, reactions, prefix, steps=2):
    '''
    Performs a series of reactions on an input compound and returns the resulting products.
    
    Args:
        input_cpd (dict): The input compound dictionary.
        reactions (list): A list of reaction dictionaries to apply sequentially.
        
    Returns:
        dict: A dict of step: product dictionaries resulting from the reactions.
    '''
    predicted_products = {}
    step_products = [input_cpd]
    counter = 0
    for step in range(steps):
        step_num = step + 1
        new_products = []
        for product in step_products:
            for reaction in reactions:
                new_product = do_reaction(product, reaction, counter, prefix)
                counter += 1
                if new_product is not None:
                    new_product['step'] = step_num
                    new_products.append(new_product)
        print(f"Step {step_num}: {len(new_products)} products.")
        step_products = new_products
        predicted_products[step_num] = new_products
        
    return predicted_products

def retrieve_rxn_sequence(product):
    '''
    Retrieves the sequence of reactions that led to a given product.
    
    Args:
        product (dict): The product dictionary, e.g. 
                    {'id': 'product_308789',
                    'mz': 515.233614272,
                    'rtime': 0,
                    'formula_dict': {'H': 25, 'O': 4, 'C': 16},
                    'formula': 'C16H25O4',
                    'input_cpd': {'id': 'product_1806',
                        'mz': 511.238699652,
                        'rtime': 0,
                        'formula_dict': {'H': 25, 'C': 17, 'O': 3},
                        'formula': 'C17H25O3',
                                'input_cpd': {'id': 'product_8',
                                'mz': 509.223049652,
                                'rtime': 0,
                                'formula_dict': {'H': 23, 'O': 3, 'C': 17},
                                'formula': 'C17H23O3',
                                        'input_cpd': {'mz': 504.22031,
                                            'formula': 'C18H22O2',
                                            'formula_dict': {'C': 18, 'H': 22, 'O': 2},
                                            'rtime': 1099,
                                            'id': 'Estrone'},
                                        'reaction': {'massDelta_signed': 5.002739652,
                                            'abs_mass': 5.002739652,
                                            'formula_difference': '+HO-C',
                                            'formula_dict': {'H': 1, 'O': 1, 'C': -1},
                                            'reactionIDs': "['HYDROLISIS_OF_ARYL_ACYLAMIDE']",
                                            'source': 'SZ_only'},
                                        'step': 1},
                                'reaction': {'massDelta_signed': 2.01565,
                                'abs_mass': 2.01565,
                                'formula_difference': 'H2',
                                'formula_dict': {'H': 2},
                                'reactionIDs': 'Forming or close double bond',
                                'source': 'MY_only'},
                                'step': 2},
                        'reaction': {'massDelta_signed': 3.99491462,
                        'abs_mass': 3.99491462,
                        'formula_difference': '+O-C',
                        'formula_dict': {'O': 1, 'C': -1},
                        'reactionIDs': "['N_DEALKYLATION_OF_SECONDARY_CARBOXAMIDES']",
                        'source': 'SZ_only'},
                    'step': 3}

    Returns:
        list: A list of cpds in the sequence leading to the product.
    '''
    sequence = []
    input_cpd = product.get('input_cpd', None)
    while input_cpd:
        sequence.append(input_cpd['id'])
        product = input_cpd
        input_cpd = input_cpd.get('input_cpd', None)
        
    sequence.reverse()  # reverse to get the correct order from input to final product
    return sequence


def get_feature_list(tsv_file):
    # Implementation for getting feature list from TSV file
    features = []
    for line in open(tsv_file).readlines()[1:]:
        a = line.split('\t')
        features.append({
            'id': a[0],
            'mz': float(a[1]),
            'rtime': float(a[2])
        })
    return features

def filter_prediction_by_detection(dict_predicted_products, list_features, mz_ppm=10, rt_tolerance=3000000):
    '''Keep only prediction end productss that are found among detected features.
    Add detection score per product, as number of matching detected features, including intermediaries.
    '''
    new_dict = {}
    for step, predicted_products in dict_predicted_products.items():
        _d = {product['id']: product for product in predicted_products}
        matched = mf.list_match_lcms_features(predicted_products, 
                                                       list_features, 
                                                       mz_ppm=mz_ppm, rt_tolerance=rt_tolerance)
        new_dict[step] = [_d[x] for x in matched]
    # do detection score; because all intermediaries are from prior steps, new_dict values have complete detected products
    detected_products = []
    for step, products in new_dict.items():
        detected_products.extend([x['id'] for x in products])
    for step, products in new_dict.items():
        for prod in products:
            seq_cpds = retrieve_rxn_sequence(prod)[1:]
            prod['detection_score'] = 1 + len([f for f in seq_cpds if f in detected_products])
    return new_dict
        

In [ ]:
sz_rxns_tsv = 'input/Merged_reactions_mass_formula_difference_SZ_10192026.tsv'
sz_rxns = []
# massDelta_signed	abs_mass	formula_difference	reactionIDs	source
for index, row in pd.read_csv(sz_rxns_tsv, sep='\t').iterrows():
    sz_rxns.append({
        'massDelta_signed': row['massDelta_signed'],
        'abs_mass': row['abs_mass'],
        'formula_difference': row['formula_difference'],
        'formula_dict': parse_chemformula_dict_signed(row['formula_difference']),
        'reactionIDs': row['reactionIDs'],  
        'source': row['source']
    })
sz_rxns[:3]

## generate predicted metabolites by substrate

In [ ]:
# ~1.5 minute
outdir = 'results/step4_predicted/'
indir = 'results/step1_filtered_results'

estrone_3step_products = multi_step_reactions(
    substrates['Estrone'], sz_rxns, prefix='estrone', steps=3)  #18 secs

detected = get_feature_list(f'{indir}/01_v7_estrone_DnCl_pellets_treatment_enriched_deuterium_matches.tsv') + get_feature_list(
    f'{indir}/01_v7_estrone_DnCl_supernatant_treatment_enriched_deuterium_matches.tsv'
)

estrone_3step_products = filter_prediction_by_detection(estrone_3step_products, detected)
json.dump(estrone_3step_products, open(os.path.join(outdir, 'estrone_3step_products.json'), 'w'), indent=4)

In [ ]:
len(estrone_3step_products[2])

In [ ]:
# estradiol
estradiol_3step_products = multi_step_reactions(
    substrates['Estradiol'], sz_rxns, prefix='estradiol', steps=3)  # 18 secs

detected = get_feature_list(f'{indir}/01_v7_estradiol_DnCl_pellets_treatment_enriched_deuterium_matches.tsv') + get_feature_list(
    f'{indir}/01_v7_estradiol_DnCl_supernatant_treatment_enriched_deuterium_matches.tsv'
)

estradiol_3step_products = filter_prediction_by_detection(estradiol_3step_products, detected)
json.dump(estradiol_3step_products, open(os.path.join(outdir, 'estradiol_3step_products.json'), 'w'), indent=4)

In [ ]:
# dhea
dhea_3step_products = multi_step_reactions(
    substrates['DHEA'], sz_rxns, prefix='dhea', steps=3)  #20 secs

detected = get_feature_list(f'{indir}/01_v7_dhea_DnHz_pellets_treatment_enriched_deuterium_matches.tsv') + get_feature_list(
    f'{indir}/01_v7_dhea_DnHz_supernatant_treatment_enriched_deuterium_matches.tsv'
)

dhea_3step_products = filter_prediction_by_detection(dhea_3step_products, detected)
json.dump(dhea_3step_products, open(os.path.join(outdir, 'dhea_3step_products.json'), 'w'), indent=4)

In [ ]:
# 4 minutes
pregnenolone_3step_products = multi_step_reactions(
    substrates['Pregnenolone'], sz_rxns, prefix='pregnenolone', steps=3)  #18 secs

detected = get_feature_list(f'{indir}/01_v7_pregnenolone_DnHz_pellets_treatment_enriched_deuterium_matches.tsv') + get_feature_list(
    f'{indir}/01_v7_pregnenolone_DnHz_supernatant_treatment_enriched_deuterium_matches.tsv'
)

pregnenolone_3step_products = filter_prediction_by_detection(pregnenolone_3step_products, detected)
json.dump(pregnenolone_3step_products, open(os.path.join(outdir, 'pregnenolone_3step_products.json'), 'w'), indent=4)

In [ ]:
# 1 minute 
progesterone_3step_products = multi_step_reactions(
    substrates['Progesterone'], sz_rxns, prefix='progesterone', steps=3)  #20 secs

detected = get_feature_list(f'{indir}/01_v7_progesterone_DnHz_pellets_treatment_enriched_deuterium_matches.tsv') + get_feature_list(
    f'{indir}/01_v7_progesterone_DnHz_supernatant_treatment_enriched_deuterium_matches.tsv'
)

progesterone_3step_products = filter_prediction_by_detection(progesterone_3step_products, detected)
json.dump(progesterone_3step_products, open(os.path.join(outdir, 'progesterone_3step_products.json'), 'w'), indent=4)

## Summary

We have generated in silico prediction of steroid metabolites. 
Each predicted product is checked for presence in experimental data; a detection score is assigned based how many compounds, including rxn intermediaries, are in detected data (deu labeling required). 

These results will be used in the next step to reconstruct reaction networks. 